# QLoRA SFT — Llama 3.2 3B Instruct para saúde da mulher

Treina adapters LoRA sobre o `meta-llama/Llama-3.2-3B-Instruct` usando o dataset SFT gerado em `preparando-fine-tuning/gerar_dataset_sft.ipynb`.


In [ ]:
!pip install -q -U "transformers>=4.46,<4.50" "peft>=0.13,<0.15" "trl>=0.12,<0.14" \
    "accelerate>=1.1,<2.0" "bitsandbytes>=0.45.0" "triton>=3.0" \
    datasets python-dotenv

In [ ]:
import os, json, torch
from pathlib import Path
from datetime import datetime
from huggingface_hub import login
from google.colab import drive
from dotenv import load_dotenv
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig

In [ ]:
drive.mount('/content/drive', force_remount=True)
DRIVE_BASE = '/content/drive/MyDrive/AssistenteHospitalar'
ENV_PATH   = f'{DRIVE_BASE}/.env'

if not load_dotenv(ENV_PATH):
    raise FileNotFoundError(f'.env não encontrado em {ENV_PATH}')
HF_TOKEN = os.getenv('HF_TOKEN')
if not HF_TOKEN:
    raise ValueError('HF_TOKEN ausente no .env')
login(token=HF_TOKEN)
print('HuggingFace autenticado.')

In [ ]:
# Pre-flight: verifica que dataset existe, GPU disponível, memória limpa
from pathlib import Path
import torch

DS_DIR = Path('/content/drive/MyDrive/AssistenteHospitalar/files/sft')
for nome in ['sft_train.jsonl', 'sft_val.jsonl', 'sft_test.jsonl']:
    p = DS_DIR / nome
    if not p.exists():
        raise FileNotFoundError(f'Faltando: {p}. Rode primeiro gerar_dataset_sft.ipynb.')
    n_linhas = sum(1 for _ in open(p, encoding='utf-8'))
    print(f'  {nome:<20} {n_linhas:>5} exemplos  ({p.stat().st_size/1024:.1f} KB)')

if not torch.cuda.is_available():
    raise RuntimeError('GPU não disponível. Runtime → Change runtime type.')

free, total = torch.cuda.mem_get_info()
print(f'\nGPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM livre: {free/1e9:.1f} / {total/1e9:.1f} GB')
if free < 10e9:
    print('⚠️  VRAM livre <10GB. Considere Runtime → Restart runtime antes de treinar.')
else:
    print('✅ Pre-flight ok.')

In [ ]:
MODEL_ID    = 'meta-llama/Llama-3.2-3B-Instruct'
DATASET_DIR = Path(f'{DRIVE_BASE}/files/sft')
TRAIN_FILE  = DATASET_DIR / 'sft_train.jsonl'
VAL_FILE    = DATASET_DIR / 'sft_val.jsonl'
TEST_FILE   = DATASET_DIR / 'sft_test.jsonl'

# Diretório de saída (separado por timestamp para não sobrescrever runs)
RUN_TAG     = datetime.now().strftime('%Y%m%d_%H%M')
OUT_BASE    = Path(f'{DRIVE_BASE}/files/finetune')
OUT_DIR     = OUT_BASE / f'llama32-3b-saude-mulher_{RUN_TAG}'
FINAL_ADAPTER_DIR = OUT_DIR / 'adapter_final'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Hiperparâmetros
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
MAX_SEQ_LEN = 2048
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 8       # batch efetivo = 16
EPOCHS = 3
LR = 2e-4
WARMUP_RATIO = 0.03

print('Run tag :', RUN_TAG)
print('Output  :', OUT_DIR)
print('Adapter :', FINAL_ADAPTER_DIR)

In [ ]:
ds = load_dataset('json', data_files={
    'train':      str(TRAIN_FILE),
    'validation': str(VAL_FILE),
    'test':       str(TEST_FILE),
})
print(ds)
print('\nExemplo train[0]:')
print(json.dumps(ds['train'][0], indent=2, ensure_ascii=False)[:600])

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    attn_implementation='eager',
)
model.config.use_cache = False
model.config.pretraining_tp = 1
model = prepare_model_for_kbit_training(model)
print('Modelo carregado em 4-bit.')
print(f'Parâmetros totais: {sum(p.numel() for p in model.parameters())/1e9:.2f}B')

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        'q_proj','k_proj','v_proj','o_proj',
        'gate_proj','up_proj','down_proj',
    ],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
training_args = SFTConfig(
    output_dir=str(OUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=WARMUP_RATIO,
    optim='paged_adamw_8bit',
    bf16=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=20,
    save_strategy='steps',
    save_steps=40,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    seed=42,
    # SFT-específicos — sem dataset_text_field (auto-detecta 'messages')
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)
print('SFTConfig pronto.')

In [ ]:
# Aplica o chat template do Llama 3 para criar o campo 'text' que o SFTTrainer espera
def render_chat(example):
    return {
        'text': tokenizer.apply_chat_template(
            example['messages'],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

ds_train = ds['train'].map(render_chat, remove_columns=ds['train'].column_names)
ds_val   = ds['validation'].map(render_chat, remove_columns=ds['validation'].column_names)

print('Exemplo renderizado:')
print(ds_train[0]['text'][:400])
print('...\n')

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,       # nome novo a partir do TRL 0.12
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
)
print('Trainer pronto. Train size:', len(ds_train), '| Val:', len(ds_val))

In [ ]:
train_result = trainer.train()
print('\nTreino concluído.')
print(train_result.metrics)

In [ ]:
# Salva apenas o adapter LoRA (poucos MB) + tokenizer + meta do treino
trainer.save_model(str(FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))

meta = {
    'base_model': MODEL_ID,
    'run_tag':    RUN_TAG,
    'epochs':     EPOCHS,
    'learning_rate': LR,
    'lora': {'r': LORA_R, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT},
    'effective_batch': PER_DEVICE_BATCH * GRAD_ACCUM,
    'train_size': len(ds['train']),
    'val_size':   len(ds['validation']),
    'metrics':    {k: float(v) for k, v in train_result.metrics.items() if isinstance(v, (int, float))},
}
with open(FINAL_ADAPTER_DIR / 'training_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print('Adapter salvo em:', FINAL_ADAPTER_DIR)

In [ ]:
# Eval final no test set
ds_test = ds['test'].map(render_chat, remove_columns=ds['test'].column_names)
eval_metrics = trainer.evaluate(ds_test)
print('\nMétricas no test set:')
for k, v in eval_metrics.items():
    print(f'  {k:<30} {v:.4f}' if isinstance(v, float) else f'  {k:<30} {v}')

with open(FINAL_ADAPTER_DIR / 'test_metrics.json', 'w', encoding='utf-8') as f:
    json.dump({k: float(v) if isinstance(v, (int, float)) else v for k, v in eval_metrics.items()},
              f, indent=2, ensure_ascii=False)

In [ ]:
# Smoke test: gera resposta com o adapter recém-treinado
model.eval()
exemplo = ds['test'][0]
msg_in = exemplo['messages'][:2]   # system + user (sem o gold assistant)
gold   = exemplo['messages'][2]['content']

inputs = tokenizer.apply_chat_template(
    msg_in, add_generation_prompt=True, return_tensors='pt', return_dict=True,
).to(model.device)
with torch.no_grad():
    out = model.generate(
        **inputs, max_new_tokens=384, do_sample=False, pad_token_id=tokenizer.eos_token_id,
    )
resposta = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print('USER:    ', msg_in[1]['content'])
print('\nFINE-TUNED:\n', resposta)
print('\nGOLD:\n', gold)